In [1]:
#Standard libraries
import numpy as np
import math as m
from scipy.constants import c
import h5py

#custom libraries
from Reconal import ray_utils, defs
from rayReader import *

/home/mjmarsee/RNOG/solarFlareAnalysis/venvSolar/lib/python3.10/site-packages/numpy/core/getlimits.py:542: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [ ]:
##CONFIG

#Geometry
tx = [0,0] #fixed point on surface of ice, keep at origin

r_step = 0.01 #dr of traced ray
d_theta = 0.01 #angular separation between rays


r_range = rmin, rmax = 0,100
z_range = zmin, zmax = -100,0


#Ice
ior = defs.ior_exp3
grad_ior = defs.grad_ior_exp3

#file containing generated rays, sorted by launch angle
outfile_name = 'ray_dictionary.h5'


In [3]:
def getLaunchAngle(arr_r,arr_z):
    length_r = np.round(arr_r[1]-arr_r[0],10)
    length_z = np.round(arr_z[1]-arr_z[0],10)
    theta_offset = 0
    if length_z<=0:
        theta_offset = 90
    launch_angle = np.rad2deg(np.arctan(np.abs(length_z/length_r))) + theta_offset
    return launch_angle


def generate_ray(launch_angle_deg, source_location, r_step=0.01, d_theta=0.01):
    theta_min,theta_max = launch_angle_deg - d_theta, launch_angle_deg + d_theta
    ray_mesh = np.linspace(theta_min, theta_max, 5)

    #Generating Rays
    rays, turnover = ray_utils.get_rays(source_location, ior, grad_ior, rmax, zmin, zmax, ray_mesh, r_step)

    precision = int(-m.log10(d_theta))
    launch_angles = np.array([np.round(getLaunchAngle(ray[0], ray[1]), precision) for ray in rays])


    sri = np.argmin(np.abs(launch_angles-launch_angle_deg))

    return rays[sri]

In [ ]:


scale = int(round(1 / d_theta))

# exact grid in integer space
i_vals = np.arange(90 * scale, 180 * scale + 1, dtype=np.int32)
launch_angle_set = i_vals / scale  # numeric values for physics

angle_labels = np.array(
    [f"{i/scale:.2f}" for i in i_vals],
    dtype=h5py.string_dtype(encoding="utf-8")
)
N_rays = len(launch_angle_set)


In [ ]:

with h5py.File(outfile_name, "w") as ray_dictionary:

    dset = ray_dictionary.create_dataset(
        "rays",
        shape=(len(launch_angle_set), 3, 100000),
        dtype=np.float32,
        chunks=(1, 3, 100000),
        compression="lzf"
    )

    valid_mask = np.zeros(len(launch_angle_set), dtype=np.bool_)

    for i, angle in enumerate(launch_angle_set):
        if angle>=179:
            print(f'Skip near-vertical ray ({angle})')
            continue
        try:
            ray = generate_ray(
                angle,
                source_location=tx,
                r_step=r_step,
                d_theta=d_theta
            )
            print(f'Ray ({angle})[{angle_labels[i]}] finished')

            dset[i] = ray.astype(np.float32)
            valid_mask[i] = True

        except Exception as e:
            print(f"FAILED {i} (angle {angle:.2f}): {e}")

        if (i + 1) % 100 == 0:
            print(f"{i+1}/{len(launch_angle_set)} rays generated")

    # store numeric + human-readable representations
    ray_dictionary.create_dataset(
        "launch_angle_labels",
        data=angle_labels
    )

    ray_dictionary.create_dataset(
        "valid_mask",
        data=valid_mask
    )

Ray (90.0)[90.00] finished
Ray (90.5)[90.50] finished
Ray (91.0)[91.00] finished
Ray (91.5)[91.50] finished
Ray (92.0)[92.00] finished
Ray (92.5)[92.50] finished
Ray (93.0)[93.00] finished
Ray (93.5)[93.50] finished
Ray (94.0)[94.00] finished
Ray (94.5)[94.50] finished
Ray (95.0)[95.00] finished
Ray (95.5)[95.50] finished
Ray (96.0)[96.00] finished
Ray (96.5)[96.50] finished
Ray (97.0)[97.00] finished
Ray (97.5)[97.50] finished
Ray (98.0)[98.00] finished
Ray (98.5)[98.50] finished
Ray (99.0)[99.00] finished
Ray (99.5)[99.50] finished
Ray (100.0)[100.00] finished
Ray (100.5)[100.50] finished
Ray (101.0)[101.00] finished
Ray (101.5)[101.50] finished
Ray (102.0)[102.00] finished
Ray (102.5)[102.50] finished
Ray (103.0)[103.00] finished
Ray (103.5)[103.50] finished
Ray (104.0)[104.00] finished
Ray (104.5)[104.50] finished
Ray (105.0)[105.00] finished
Ray (105.5)[105.50] finished
Ray (106.0)[106.00] finished
Ray (106.5)[106.50] finished
Ray (107.0)[107.00] finished
Ray (107.5)[107.50] finis